In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

table_name = "silver_sales"

df_silver = spark.range(10).withColumn("sales_amount", F.lit(100))

if spark.catalog.tableExists(table_name):
    delta_table = DeltaTable.forName(spark, table_name)

    delta_table.alias("target").merge(
        df_silver.alias("source"),
        "target.id = source.id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

    print("✅ Upsert completed")
else:
    df_silver.write.saveAsTable(table_name)
    print("✅ Initial load complete")
